In [1]:
import pandas as pd

file_path = "House Sales by County.xlsx"
data = pd.read_excel(file_path, sheet_name=None)

In [2]:
def analyse_house(data, county, asking_price, income, deposit, rate, years=30):

    df = data[county].copy()
    df["date"] = pd.to_datetime(df["date"])

    # --- Market stats ---
    median_price = df["price"].median()
    lower_range = df["price"].quantile(0.25)
    upper_range = df["price"].quantile(0.75)

    # --- Price classification ---
    if asking_price > upper_range:
        price_status = "overpriced"
    elif asking_price < lower_range:
        price_status = "cheap"
    else:
        price_status = "typical"

    # --- Mortgage ---
    loan = asking_price - deposit
    monthly_rate = rate / 100 / 12
    months = years * 12

    payment = loan * (monthly_rate * (1 + monthly_rate) ** months) / ((1 + monthly_rate) ** months - 1)
    monthly_income = income / 12
    ratio = payment / monthly_income

    # --- Affordability ---
    if ratio > 0.40:
        affordability = "high pressure"
    elif ratio > 0.30:
        affordability = "moderate"
    else:
        affordability = "comfortable"

    # --- Score ---
    score = 100

    if asking_price > upper_range:
        score -= 30
    elif asking_price > median_price:
        score -= 15

    if ratio > 0.40:
        score -= 40
    elif ratio > 0.30:
        score -= 20

    score = max(0, min(score, 100))

    # --- Decision ---
    if asking_price > upper_range and ratio > 0.40:
        decision = "High risk"
    elif asking_price > upper_range:
        decision = "Overpriced"
    elif ratio > 0.40:
        decision = "Unaffordable"
    else:
        decision = "Reasonable"

    # --- Human-friendly explanation ---
    if price_status == "overpriced":
        price_text = "This property is priced above the typical range for the area."
    elif price_status == "cheap":
        price_text = "This property is priced below the typical range, which may indicate a good deal or underlying issues."
    else:
        price_text = "This property is priced in line with the local market."

    if ratio > 0.40:
        affordability_text = "Your mortgage payments would take up a large portion of your income, which could be financially risky."
    elif ratio > 0.30:
        affordability_text = "Your mortgage payments are somewhat high relative to your income, so this may feel financially tight."
    else:
        affordability_text = "Your mortgage payments look comfortable relative to your income."

    if score >= 80:
        score_text = "This looks like a strong overall purchase."
    elif score >= 60:
        score_text = "This looks like a reasonable purchase, but there are some risks to consider."
    elif score >= 40:
        score_text = "This purchase carries some notable risks."
    else:
        score_text = "This looks like a high-risk purchase."

    summary = f"""
{price_text}

{affordability_text}

Monthly payment: £{round(payment)}
This represents {round(ratio*100)}% of your monthly income.

Overall assessment: {score_text}
"""

    return {
        "decision": decision,
        "score": score,
        "summary": summary
    }

In [3]:
result = analyse_house(
    data=data,
    county="WEST SUSSEX",
    asking_price=450000,
    income=90000,
    deposit=80000,
    rate=5.2
)

print(result["summary"])


This property is priced in line with the local market.

Your mortgage payments look comfortable relative to your income.

Monthly payment: £2032
This represents 27% of your monthly income.

Overall assessment: This looks like a strong overall purchase.

